# Costhet cost calibration (re-run, this time keeping `data/raw/`)

Settings: **Accelerator = CPU only**, **Internet = ON** (for `pip install`). Add the dataset
`kaggle_bundle.zip` (uploaded separately as a Kaggle Dataset) as this notebook's input
before running. Do the whole run in **one session** (Kaggle can swap the underlying CPU across sessions).

This reproduces `data/profile_baseline/profiles.json`, `data/derived/labels.json`,
`data/manifest_static/manifest_static.json`, and this time also zips the 1044 write-once
`data/raw/<event_id>/<test_id>/result.json` artifacts so gate 6 (`label_reproducibility`) can be
independently re-verified back in the paper repo.

In [ ]:
# Cell 1 -- pinned, quiet environment
# cart/cli.py imports PyYAML directly, and it is NOT a Qiskit dependency, so install it explicitly
# (this is the fix for the first failure: ModuleNotFoundError: No module named 'yaml').
import os
for k, v in {"RAYON_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "QISKIT_IN_PARALLEL": "TRUE"}.items():
    os.environ[k] = v
!pip -q install qiskit==2.4.2 pyyaml numpy
import platform, subprocess
print("python", platform.python_version())
print(subprocess.run(["bash", "-lc", "grep -m1 'model name' /proc/cpuinfo; nproc"],
                      capture_output=True, text=True).stdout)  # <-- RECORD THIS for the paper's reproducibility statement

In [ ]:
# Cell 2 -- stage the project from the attached dataset, robustly.
# Kaggle sometimes auto-extracts an uploaded .zip dataset, sometimes it leaves it as a .zip file inside
# the dataset folder. Handle both: if we find pyproject.toml directly, use that folder; otherwise look
# for a .zip anywhere under /kaggle/input and unzip it ourselves. Then verify src/cart is really there
# before moving on (this is the fix for the second failure: ModuleNotFoundError: No module named 'cart').
import glob, os, shutil, zipfile

!mkdir -p /kaggle/working/proj

proj_root = None

# Case A: already extracted somewhere under /kaggle/input -- find pyproject.toml
hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
if hits:
    proj_root = os.path.dirname(hits[0])
    print("found extracted project at:", proj_root)
else:
    # Case B: still a .zip inside the dataset folder -- unzip it ourselves
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    print("no extracted pyproject.toml found; zip candidates:", zips)
    assert zips, "No dataset found under /kaggle/input at all -- did you attach kaggle_bundle.zip as this notebook's input?"
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall("/kaggle/working/proj")
    proj_root = "/kaggle/working/proj"
    print("unzipped", zips[0], "into", proj_root)

if proj_root != "/kaggle/working/proj":
    !cp -r {proj_root}/* /kaggle/working/proj/

%cd /kaggle/working/proj
!ls -la
assert os.path.isdir("src/cart"), "src/cart still missing after staging -- inspect the ls -la output above"
print("OK: src/cart is present")
import sys; sys.path.insert(0, "src")

In [ ]:
# Cell 3 -- calibration: fresh raw, 116-unit costhet corpus, >=5-run warmup-discarded median
!rm -rf data/raw && mkdir -p data/raw data/derived data/profile_baseline data/manifest_static
!PYTHONPATH=src python -m cart.cli manifest --profile costhet
!PYTHONPATH=src python -m cart.cli ground-truth --unit-limit 116 --cost-repeats 5
# Sanity check before Cell 4 -- fail loudly here instead of a confusing FileNotFoundError later
import os
assert os.path.exists("data/profile_baseline/profiles.json"), \
    "profiles.json was not produced -- scroll up in this cell's output for the actual error from manifest/ground-truth"

In [ ]:
# Cell 4 -- surface the deliverables, INCLUDING the raw write-once artifacts this time
# (gate 6, label_reproducibility, checks that every label's raw data/raw/<event_id>/<test_id>/result.json
#  exists and matches -- the first Kaggle round only downloaded the three derived files below, so
#  data/raw/ (1044 small JSON files, ~3 MB total) was left behind and the gate could not be
#  independently re-verified. Zip it this time.)
import shutil, os
shutil.make_archive("data_raw_costhet", "zip", "data/raw")
for f in ["data/profile_baseline/profiles.json",
          "data/derived/labels.json",
          "data/manifest_static/manifest_static.json",
          "data_raw_costhet.zip"]:
    print(f, os.path.getsize(f), "bytes")

In [ ]:
# Cell 5 (optional sanity check, still on Kaggle) -- gates should be 6/6 right here since data/raw/ is fresh
!PYTHONPATH=src python -c "from cart.gates.validity import run_all_gates; import json; print(json.dumps(run_all_gates(), indent=2))"

## After this run

Download from the Kaggle output panel:
1. `data/profile_baseline/profiles.json`
2. `data/derived/labels.json`
3. `data/manifest_static/manifest_static.json`
4. `data_raw_costhet.zip`

Bring all four back, plus the printed **CPU model / core count / Python version** from Cell 1. Unzip
`data_raw_costhet.zip` into `data/raw/` in the paper repo (merges with the existing historical-event
artifacts already there). Then in the repo: `python3 -c "from cart.gates.validity import run_all_gates; import json; print(json.dumps(run_all_gates(), indent=2))"` to confirm 6/6 before it goes into the manuscript (§6.2).